# Confound Probe — Information Leakage Diagnostic

**This is a validation/diagnostic notebook, not a production model.**

It tests a specific threat to the central claim of the embedding project: does the final
128-d embedding space (from `04_train_encoder.ipynb`) encode information about **palo** or
**recording origin** (TONAS vs. cante100) that a linear classifier can exploit?

If a simple logistic regression can predict palo or source well above chance from the
embeddings alone, it means the space carries information beyond cantaor identity — a
relevant qualification for the main result.

This notebook:
- Does **not** retrain the embedding model.
- Does **not** save any model artefacts.
- Uses only `data/processed/final_results.npz` (read-only) and the two raw catalogues for metadata.
- Is entirely independent of any other classification project in the portfolio.

In [ ]:
import os

# Notebooks use relative paths — run from the project root (cante-embeddings/).
if not os.path.exists('data/processed'):
    raise RuntimeError("Run this notebook from the project root (cante-embeddings/).")

## 1. Load embeddings and reconstruct per-recording metadata

`final_results.npz` stores the 128-d recording-level embeddings but only carries
`recording_id`, `norm_name`, and `split`. We reconstruct `source` (TONAS / cante100)
from `train_test_split.csv` and `palo` from the two raw catalogues via a filename join.

In [2]:
import numpy as np
import pandas as pd

# --- embeddings ---
results = np.load('data/processed/final_results.npz', allow_pickle=True)
embeddings   = results['embeddings']      # (109, 128)
recording_id = results['recording_id']
norm_name    = results['norm_name']
split        = results['split']

meta = pd.DataFrame({
    'recording_id': recording_id,
    'norm_name':    norm_name,
    'split':        split,
})

# --- source (TONAS / cante100) ---
split_df = pd.read_csv('data/processed/train_test_split.csv')
meta = meta.merge(split_df[['recording_id', 'source']], on='recording_id', how='left')

# --- palo from cante100 catalogue ---
c100_cat = pd.read_csv(
    'data/raw/cante100_catalogo - cante100_catalogo.csv.csv'
)[['filename', 'palo_normalizado']].rename(
    columns={'filename': 'recording_id', 'palo_normalizado': 'palo'}
)

# --- palo from TONAS catalogue ---
tonas_cat = pd.read_csv(
    'data/raw/tonas_catalogo - tonas_catalogo.csv.csv'
)[['archivo_wav', 'palo']].rename(
    columns={'archivo_wav': 'recording_id'}
)

palo_map = pd.concat([c100_cat, tonas_cat], ignore_index=True)
meta = meta.merge(palo_map, on='recording_id', how='left')

print(f"Rows in meta: {len(meta)}  |  Embeddings shape: {embeddings.shape}")
print(f"Source counts:\n{meta['source'].value_counts()}")
print(f"\nRows with missing palo: {meta['palo'].isna().sum()}")
meta[meta['palo'].isna()][['recording_id', 'source', 'palo']]

Rows in meta: 109  |  Embeddings shape: (109, 128)
Source counts:
source
cante100    74
tonas       35
Name: count, dtype: int64

Rows with missing palo: 1


,recording_id,source,palo
68,092_AntonioMairenaTonas.mp3,cante100,NaN


### Fix missing palo for track 092

`092_AntonioMairenaTonas.mp3` has an empty `palo_normalizado` in the cante100 catalogue
(documented there as "palo no reconocido en mapa, revisar" — the filename field
dropped the underscore separator so the catalogue parser missed the palo token).
The recording is unambiguously a *Tonás* performance by Antonio Mairena; the filename
itself, the series context (tracks 090-100 are the Tonás section), and the cantaor's
known repertoire all confirm this. We assign it manually.

In [3]:
meta.loc[meta['recording_id'] == '092_AntonioMairenaTonas.mp3', 'palo'] = 'Tonás'

print(f"Missing palo after fix: {meta['palo'].isna().sum()}")
print("\nPalo distribution — cante100 recordings (eligible set):")
c100_mask = meta['source'] == 'cante100'
palo_counts = meta.loc[c100_mask, 'palo'].value_counts().sort_values()
print(palo_counts.to_string())
print(f"\nMin class: '{palo_counts.index[0]}' = {palo_counts.iloc[0]} samples")
print(f"Total cante100 recordings for palo probe: {c100_mask.sum()}")

Missing palo after fix: 0

Palo distribution — cante100 recordings (eligible set):
palo
Bulerías                                4
Cantes de Ida y Vuelta (Americanos)     6
Malagueñas-Granaínas                    6
Tonás                                   7
Fandangos                               8
Soleares                                8
Seguiriyas                              8
Alegrías                                8
Cantes Mineros                          9
Tientos-Tangos                         10

Min class: 'Bulerías' = 4 samples
Total cante100 recordings for palo probe: 74


## 2. Cross-validation strategy

**Palo probe (cante100 only, n=74):**  
Bulerías has 4 samples — the minimum across 10 classes. `StratifiedKFold(k)` requires
at least `k` samples per class, so `k=4` is the ceiling, but that leaves only 1 Bulerías
sample in each test fold and 3 in train — a very noisy estimate per fold.  
We use **LeaveOneOut (LOO)** instead: each fold has exactly 1 test sample and 73 in
train, regardless of class size. LOO maximises training data in each fold and is the
standard recommendation for n < 100. The LOO score is equivalent to the fraction of
samples classified correctly, averaged over all 74 single-sample test sets.  
Metric: `balanced_accuracy` (averages per-class recall — necessary because
Bulerías=4 vs. Tientos-Tangos=10).

**Source probe (all recordings, n=109):**  
Two classes: cante100 (74) and TONAS (35). Min class = 35 → `StratifiedKFold(5)` is
fine (7 TONAS samples per test fold). We use `k=5` here rather than LOO to keep
compute tractable for the permutation test (500 × 5 = 2,500 fits vs. 500 × 109 = 54,500).  
Metric: `balanced_accuracy`.

**Permutation test (both probes):**  
We use `sklearn.model_selection.permutation_test_score`, which globally shuffles the
label vector once per permutation and then runs the full CV procedure — the correct
implementation. Shuffling inside each fold would be wrong: it would destroy global label
structure that only the full-dataset permutation is supposed to destroy.
n_permutations=500; p-value = (# perms where perm_score ≥ real_score + 1) / 501.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    LeaveOneOut, StratifiedKFold,
    cross_val_score, permutation_test_score
)

clf_kwargs = dict(
    C=0.1,           # strong L2: 128-d features, only ~7 samples per class
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
)

clf   = LogisticRegression(**clf_kwargs)
loo   = LeaveOneOut()
skf5  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
N_PERM = 500

## 3. Probe (a): Palo — cante100 recordings only

Question: is palo (the flamenco form) linearly decodable from the 128-d embedding?

Restricted to cante100 because all TONAS recordings belong to the tonás family
(Debla / Martinete) — including them would confound palo with recording origin.

In [5]:
X_palo = embeddings[c100_mask.values]          # (74, 128)
y_palo = meta.loc[c100_mask, 'palo'].values    # 10-class

# LOO cross-validation score
palo_cv_scores = cross_val_score(
    clf, X_palo, y_palo, cv=loo, scoring='balanced_accuracy'
)
palo_real = palo_cv_scores.mean()
palo_std  = palo_cv_scores.std()

print("=== Palo probe (cante100, LOO) ===")
print(f"Balanced accuracy: {palo_real:.3f} ± {palo_std:.3f}  (std across {len(palo_cv_scores)} LOO folds)")
print(f"Chance baseline:   0.100  (1 / 10 classes, balanced)")

=== Palo probe (cante100, LOO) ===
Balanced accuracy: 0.243 ± 0.429  (std across 74 LOO folds)
Chance baseline:   0.100  (1 / 10 classes, balanced)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/skle

In [6]:
print("Running permutation test for palo probe (500 permutations × 74 LOO folds)...")

palo_real_pt, palo_perm_scores, palo_pvalue = permutation_test_score(
    clf, X_palo, y_palo,
    cv=loo,
    scoring='balanced_accuracy',
    n_permutations=N_PERM,
    random_state=42,
    n_jobs=-1,
)

print(f"\nReal score (from permutation_test_score): {palo_real_pt:.3f}")
print(f"Null distribution — mean: {palo_perm_scores.mean():.3f}, "
      f"std: {palo_perm_scores.std():.3f}, "
      f"max: {palo_perm_scores.max():.3f}")
print(f"p-value: {palo_pvalue:.4f}")
palo_percentile = (palo_perm_scores < palo_real_pt).mean() * 100
print(f"Real score is at the {palo_percentile:.1f}th percentile of the null distribution")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/skle

Running permutation test for palo probe (500 permutations × 74 LOO folds)...


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Librar

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes n

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Librar

...

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Librar

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/skle

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/skle


Real score (from permutation_test_score): 0.243
Null distribution — mean: 0.106, std: 0.044, max: 0.230
p-value: 0.0020
Real score is at the 100.0th percentile of the null distribution


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/skle

## 4. Probe (b): Recording source — all 109 recordings

Question: is dataset origin (TONAS vs. cante100) linearly decodable from the 128-d embedding?

This is the most direct test for the acoustic-conditions confound: TONAS recordings
are a cappella, very short (8–33 s), mono WAV archival; cante100 are accompanied,
much longer (75–420 s), MP3. A model that learned to separate these domains rather than
cantaor identity would achieve high source accuracy.

In [7]:
X_source = embeddings                          # (109, 128)
y_source = meta['source'].values               # 'cante100' / 'tonas'

source_cv_scores = cross_val_score(
    clf, X_source, y_source, cv=skf5, scoring='balanced_accuracy'
)
source_real = source_cv_scores.mean()
source_std  = source_cv_scores.std()

print("=== Source probe (all 109 recordings, StratifiedKFold-5) ===")
print(f"Balanced accuracy: {source_real:.3f} ± {source_std:.3f}  (std across 5 folds)")
print(f"Chance baseline:   0.500  (balanced 2-class)")

=== Source probe (all 109 recordings, StratifiedKFold-5) ===
Balanced accuracy: 0.979 ± 0.028  (std across 5 folds)
Chance baseline:   0.500  (balanced 2-class)


In [8]:
print("Running permutation test for source probe (500 permutations × 5 folds)...")

source_real_pt, source_perm_scores, source_pvalue = permutation_test_score(
    clf, X_source, y_source,
    cv=skf5,
    scoring='balanced_accuracy',
    n_permutations=N_PERM,
    random_state=42,
    n_jobs=-1,
)

print(f"\nReal score (from permutation_test_score): {source_real_pt:.3f}")
print(f"Null distribution — mean: {source_perm_scores.mean():.3f}, "
      f"std: {source_perm_scores.std():.3f}, "
      f"max: {source_perm_scores.max():.3f}")
print(f"p-value: {source_pvalue:.4f}")
source_percentile = (source_perm_scores < source_real_pt).mean() * 100
print(f"Real score is at the {source_percentile:.1f}th percentile of the null distribution")

Running permutation test for source probe (500 permutations × 5 folds)...



Real score (from permutation_test_score): 0.979
Null distribution — mean: 0.482, std: 0.071, max: 0.677
p-value: 0.0020
Real score is at the 100.0th percentile of the null distribution


## 5. Summary and interpretation

In [9]:
print("=" * 55)
print("CONFOUND PROBE — SUMMARY")
print("=" * 55)
print(f"{'Probe':<20} {'Bal. Acc':>9} {'Baseline':>9} {'p-value':>9}")
print("-" * 55)
print(f"{'Palo (c100, LOO)':<20} {palo_real_pt:>9.3f} {'0.100':>9} {palo_pvalue:>9.4f}")
print(f"{'Source (all, k=5)':<20} {source_real_pt:>9.3f} {'0.500':>9} {source_pvalue:>9.4f}")
print("=" * 55)

CONFOUND PROBE — SUMMARY
Probe                 Bal. Acc  Baseline   p-value
-------------------------------------------------------
Palo (c100, LOO)         0.243     0.100    0.0020
Source (all, k=5)        0.979     0.500    0.0020


### Interpretation guide

| Result | Meaning |
|---|---|
| **Palo ≈ baseline (0.10), p > 0.05** | Palo is not linearly decodable. The embedding does not use palo as an organising principle. Supports the central claim. |
| **Palo >> baseline, p < 0.05** | Palo information survives in the embedding. Part of the cantaor-discrimination signal may come from the fact that each cantaor tends to perform certain palos — not from vocal style alone. The main AUC result should be qualified. |
| **Source ≈ baseline (0.50), p > 0.05** | Recording-domain differences (a cappella vs. accompanied, duration, format) are not linearly decodable. The embedding abstracts across datasets. Strong support for validity across the 10 cross-dataset cantaores. |
| **Source >> baseline, p < 0.05** | The embedding encodes acoustic/technical domain differences. The 10 cantaores present in both datasets may be partially identified because their TONAS and cante100 recordings cluster by domain rather than by voice. This is the most concerning finding for the project's central claim. |

**Note on what these probes cannot establish:**
High accuracy on either probe does not *invalidate* the main AUC result — the embedding
could simultaneously encode cantaor identity, palo, and source. Low accuracy on both
probes would be the strongest available evidence (within this dataset's limits) that the
embedding captures voice-specific style rather than incidental structure.

## 6. Pair-level breakdown: cross-dataset vs. within-dataset

Before re-evaluating, we decompose the original 2,418 evaluation pairs by whether
the test and train recordings come from the same dataset or different ones.
This quantifies how many pairs the source-confound affects directly.

In [10]:
# Re-add emb_idx explicitly so indexing into embeddings is unambiguous
meta2 = pd.DataFrame({
    'emb_idx':      range(len(recording_id)),
    'recording_id': recording_id,
    'norm_name':    norm_name,
    'split':        split,
})
meta2 = meta2.merge(split_df[['recording_id', 'source']], on='recording_id', how='left')

test_rows  = meta2[meta2['split'] == 'test'].copy()
train_rows = meta2[meta2['split'] == 'train'].copy()

# Build all original pairs
pairs = []
for _, t in test_rows.iterrows():
    t_emb = embeddings[t['emb_idx']]
    for _, tr in train_rows.iterrows():
        tr_emb = embeddings[tr['emb_idx']]
        pairs.append({
            'distance':     1 - float(np.dot(t_emb, tr_emb)),
            'label':        int(t['norm_name'] == tr['norm_name']),
            'cross_dataset': t['source'] != tr['source'],
            'test_source':  t['source'],
            'train_source': tr['source'],
        })

df_pairs = pd.DataFrame(pairs)
print(f"Total pairs: {len(df_pairs)}  "
      f"| positives: {df_pairs['label'].sum()}  "
      f"| negatives: {(df_pairs['label']==0).sum()}")
print()

for lname, lval in [("POSITIVE (same cantaor)", 1), ("NEGATIVE (diff cantaor)", 0)]:
    sub   = df_pairs[df_pairs['label'] == lval]
    n     = len(sub)
    wthin = int((~sub['cross_dataset']).sum())
    cross = int(sub['cross_dataset'].sum())
    print(f"{lname}: {n} total")
    print(f"  within-dataset:  {wthin:4d}  ({wthin/n*100:5.1f}%)")
    print(f"  cross-dataset:   {cross:4d}  ({cross/n*100:5.1f}%)")
    print()

print("Cross-dataset positive pairs by direction:")
print(df_pairs[df_pairs['label']==1].groupby(['test_source','train_source']).size().to_string())

Total pairs: 2418  | positives: 78  | negatives: 2340

POSITIVE (same cantaor): 78 total
  within-dataset:    57  ( 73.1%)
  cross-dataset:     21  ( 26.9%)

NEGATIVE (diff cantaor): 2340 total
  within-dataset:  1306  ( 55.8%)
  cross-dataset:   1034  ( 44.2%)

Cross-dataset positive pairs by direction:
test_source  train_source
cante100     cante100        44
             tonas           12
tonas        cante100         9
             tonas           13


### Interpretation: how the source confound enters the original evaluation

With source probe accuracy 0.979, the embedding nearly perfectly separates TONAS from
cante100 recordings. This manifests in two opposing ways in the original AUC:

**Cross-dataset negatives (1,034 pairs, 44.2% of all negatives):** different cantaors,
different datasets. The model assigns high cosine distance not only because they are
different people but also because the recordings live in distinct acoustic domains.
These are systematically *easier* negatives — they inflate discrimination performance
at low-threshold operating points.

**Cross-dataset positives (21 pairs, 26.9% of all positives):** *same* cantaor,
different datasets. The source-domain gap now works *against* the model — two recordings
of the same person end up far apart in embedding space due to the acoustic-domain
difference. These are systematically *harder* positives — they deflate TPR.

The net effect on the headline AUC is therefore not a simple inflation:
the source confound introduces noise in both directions simultaneously.

## 7. Clean re-evaluation: cante100-only pairs

We restrict both the test and train pool to cante100 recordings, eliminating
the source confound entirely. Cantaores whose only train recording is in TONAS
(Rafael Romero, Santiago Donday) contribute no positive pairs here and are
absent from this evaluation — this is expected and documented.

In [11]:
from sklearn.metrics import roc_auc_score, roc_curve

c100_pairs = df_pairs[
    (df_pairs['test_source']  == 'cante100') &
    (df_pairs['train_source'] == 'cante100')
].copy()

print(f"cante100-only pairs:  {len(c100_pairs)}")
print(f"  positives: {c100_pairs['label'].sum()}")
print(f"  negatives: {(c100_pairs['label']==0).sum()}")
print(f"  note: 2 cantaores (Rafael Romero, Santiago Donday) excluded"
      f" — their only train recordings are in TONAS")

def auc_eer(dist, labels):
    scores = -np.asarray(dist, dtype=float)
    labs   = np.asarray(labels, dtype=int)
    auc    = roc_auc_score(labs, scores)
    fpr, tpr, _ = roc_curve(labs, scores)
    fnr  = 1 - tpr
    idx  = np.nanargmin(np.abs(fpr - fnr))
    eer  = (fpr[idx] + fnr[idx]) / 2
    return auc, eer

def bootstrap_ci(dist, labels, n_boot=2000, seed=42):
    rng  = np.random.default_rng(seed)
    n    = len(dist)
    aucs, eers = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        l_b = labels[idx]
        if len(np.unique(l_b)) < 2:
            continue
        a, e = auc_eer(dist[idx], l_b)
        aucs.append(a); eers.append(e)
    return (np.percentile(aucs, [2.5, 97.5]),
            np.percentile(eers, [2.5, 97.5]))

print("\nComputing bootstrap CIs (2000 resamples each)...")
auc_o, eer_o = auc_eer(df_pairs['distance'].values, df_pairs['label'].values)
auc_c, eer_c = auc_eer(c100_pairs['distance'].values, c100_pairs['label'].values)
ci_auc_o, ci_eer_o = bootstrap_ci(df_pairs['distance'].values,  df_pairs['label'].values)
ci_auc_c, ci_eer_c = bootstrap_ci(c100_pairs['distance'].values, c100_pairs['label'].values)

cante100-only pairs:  1113
  positives: 44
  negatives: 1069
  note: 2 cantaores (Rafael Romero, Santiago Donday) excluded — their only train recordings are in TONAS

Computing bootstrap CIs (2000 resamples each)...


In [12]:
W = 24
print(f"{'Metric':<{W}} {'Original (all 109)':>20} {'cante100-only':>16}")
print("-" * (W + 38))
print(f"{'n pairs':<{W}} {len(df_pairs):>20,} {len(c100_pairs):>16,}")
print(f"{'n positives':<{W}} {int(df_pairs['label'].sum()):>20} {int(c100_pairs['label'].sum()):>16}")
print(f"{'n negatives':<{W}} {int((df_pairs['label']==0).sum()):>20} {int((c100_pairs['label']==0).sum()):>16}")
print(f"{'AUC':<{W}} {auc_o:>20.4f} {auc_c:>16.4f}")
print(f"{'AUC 95% CI':<{W}} "
      f"{'[{:.3f}, {:.3f}]'.format(*ci_auc_o):>20} "
      f"{'[{:.3f}, {:.3f}]'.format(*ci_auc_c):>16}")
print(f"{'EER':<{W}} {eer_o*100:>19.1f}% {eer_c*100:>15.1f}%")
print(f"{'EER 95% CI':<{W}} "
      f"{'[{:.1f}%, {:.1f}%]'.format(*[x*100 for x in ci_eer_o]):>20} "
      f"{'[{:.1f}%, {:.1f}%]'.format(*[x*100 for x in ci_eer_c]):>16}")
print()
print(f"delta AUC (original − c100-only): {auc_o - auc_c:+.4f}")
print(f"delta EER (original − c100-only): {(eer_o - eer_c)*100:+.1f} pp")

Metric                     Original (all 109)    cante100-only
--------------------------------------------------------------
n pairs                                 2,418            1,113
n positives                                78               44
n negatives                              2340             1069
AUC                                    0.8009           0.8132
AUC 95% CI                     [0.751, 0.848]   [0.739, 0.881]
EER                                     32.1%            25.4%
EER 95% CI                     [24.0%, 34.9%]   [17.5%, 35.6%]

delta AUC (original − c100-only): -0.0123
delta EER (original − c100-only): +6.7 pp


### How to read this table

The source confound does **not** produce a simple inflation of the headline AUC.
It introduces noise in two opposing directions:

- **Deflation via hard positives:** 21 same-cantaor pairs that span the TONAS/cante100
  domain gap. The embedding places these far apart (source probe accuracy 0.979),
  penalising TPR. Removing them *improves* AUC and EER.
- **Inflation via easy negatives:** 1,034 cross-dataset negative pairs that are trivially
  separated by domain. Removing them *hurts* AUC.

The net effect on AUC is nearly zero (Δ = −0.012, CIs fully overlapping), but the
**EER improves by 6.7 pp** on the cante100-only subset — the hard cross-dataset
positives were pulling it up disproportionately.

The deeper issue with the source confound is therefore not metric inflation but
**interpretive validity**: for the 10 cantaores present in both datasets, the model
partially relies on domain detection rather than voice identity. The within-cante100
evaluation (AUC 0.813, EER 25.4%) is the cleaner estimate of what the embedding
captures about vocal style, though its confidence intervals remain wide
(AUC 95% CI [0.739, 0.881], EER 95% CI [17.5%, 35.6%]) due to only 44 positive pairs.

## 8. Silhouette score: quantifying cantaor cluster structure

The UMAP visualisations in `04_train_encoder.ipynb` suggest that fine-tuning introduces
some cantaor-level grouping. Silhouette score provides a metric-space counterpart
independent of dimensionality reduction.

**Key design choices:**
- Labels: cantaor (`norm_name`)
- Metric: cosine distance — consistent with the training objective and evaluation protocol
- Embeddings: original-dimension (512-d pretrained, 128-d fine-tuned), **not** 2-D UMAP
  coordinates (UMAP optimises layout, not distance faithfulness)
- Exclusion: cantaores with fewer than 2 recordings in a given subset are excluded
  (silhouette requires ≥2 samples per class)
- Statistical context: 999-permutation test, shuffling cantaor labels; p = proportion of
  permuted scores ≥ observed

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score

_rng    = np.random.default_rng(42)
_N_PERM = 999

# pretrained 512-d (OpenL3 mean-pooled, no fine-tuning)
_pre     = np.load('data/processed/recording_embeddings.npz', allow_pickle=True)
_pre_df  = pd.DataFrame({'recording_id': _pre['recording_id'],
                          'pre_idx':      np.arange(len(_pre['recording_id']))})
_emb_pre_raw = _pre['embeddings']   # (109, 512)

# align pretrained row-order with meta (both 109 rows; verify)
_aligned  = meta[['recording_id', 'norm_name', 'source']].reset_index(drop=True)
_aligned  = _aligned.merge(_pre_df, on='recording_id', how='left')
assert _aligned['pre_idx'].notna().all(), 'pretrained file missing some recordings'
_emb_pre  = _emb_pre_raw[_aligned['pre_idx'].astype(int).values]
_emb_ft   = embeddings  # (109, 128), already aligned with meta
_names    = meta['norm_name'].values
_full     = np.ones(len(meta), dtype=bool)
_c100     = (meta['source'] == 'cante100').values

def _sil_block(emb, names, mask, label):
    X = emb[mask]
    y = names[mask]
    counts = pd.Series(y).value_counts()
    valid  = counts[counts >= 2].index
    keep   = np.isin(y, valid)
    X, y   = X[keep], y[keep]
    n_c, n_r = int(len(np.unique(y))), int(len(y))
    score  = float(silhouette_score(X, y, metric='cosine'))
    null   = [float(silhouette_score(X, _rng.permutation(y), metric='cosine'))
               for _ in range(_N_PERM)]
    p      = sum(s >= score for s in null) / _N_PERM
    print(f'  {label:<45}  n_c={n_c:2d}  n_r={n_r:3d}  sil={score:+.4f}  p={p:.3f}')
    return n_c, n_r, score, p

print('Silhouette score — cosine metric, cantaor labels, 999-permutation test')
print('=' * 78)
print('Full set:')
_r1 = _sil_block(_emb_pre, _names, _full, 'pretrained OpenL3 512-d')
_r2 = _sil_block(_emb_ft,  _names, _full, 'fine-tuned 128-d')
print('\ncante100-only:')
_r3 = _sil_block(_emb_pre, _names, _c100, 'pretrained OpenL3 512-d')
_r4 = _sil_block(_emb_ft,  _names, _c100, 'fine-tuned 128-d')
print('\nSummary table:')
W = 20
print(f"{'':38}  {'full set':>{W}}  {'cante100-only':>{W}}")
print(f"{'Pretrained OpenL3 512-d':<38}  {_r1[2]:>+.4f} (p={_r1[3]:.3f})  "
      f"{_r3[2]:>+.4f} (p={_r3[3]:.3f})")
print(f"{'Fine-tuned 128-d':<38}  {_r2[2]:>+.4f} (p={_r2[3]:.3f})  "
      f"{_r4[2]:>+.4f} (p={_r4[3]:.3f})")


### Interpretation

All four scores are above the null distribution (p < 0.001, 999 permutations), but the
direction of significance matters.

**The pretrained baseline score (−0.2141, p < 0.001) should not be read as evidence of
good cantaor separation.** A strongly negative silhouette with a significant permutation
p-value indicates structure that is detectable but weak: recordings are on average closer
to some other cantaor's neighbourhood than to their own, yet that pattern is still less
likely under random labels than any of 999 permutations. With 31 classes and 2–14
recordings each in 512 dimensions, substantial cluster overlap is expected — the
significance signals that some signal exists above noise, not that clusters are
well-separated.

**Fine-tuning raises the silhouette in both subsets**: −0.2141 → −0.1220 (full set,
Δ = +0.0921) and −0.1084 → +0.0259 (cante100-only, Δ = +0.1343). The improvement is
larger when the domain confound is removed, which is consistent with the source probe
result.

**The primary finding is the zero crossing in the cante100-only fine-tuned score
(+0.0259, p < 0.001).** This is the first quantitative evidence of real cantaor grouping
in the embedding space: within cante100 recordings, a recording is on average closer to the other recordings by the same cantaor
than to the recordings of the nearest other cantaor. The value is small and describes modest structure, not clean separation — but it
is above zero and above the null distribution. UMAP projections in `04_train_encoder.ipynb`
suggested some cantaor clustering visually; this score provides a metric-space counterpart
independent of dimensionality reduction.

**The domain confound is visible in the silhouette gap.** The full-set fine-tuned score
(−0.1220) sits 0.1479 points below the cante100-only score (+0.0259). The dimension that
separates TONAS from cante100 recordings (source probe: balanced accuracy 0.979) pulls
clusters in a direction that does not align with cantaor identity, increasing
within-cantaor spread and depressing the silhouette. Restricting to cante100 removes that
pressure and reveals the cantaor structure that fine-tuning introduced — a metric-space
correlate of the source probe's finding.

## Limitations and known confounds

This section documents the results of three post-hoc diagnostic analyses
run on the frozen final embeddings, without any retraining.
These are exploratory audits, not a single confirmatory test; no correction for multiple
comparisons was applied.

### Evidence summary

| Diagnostic | Result | Interpretation |
|---|---|---|
| **Source probe** (logistic regression, TONAS vs. cante100, all 109 recordings) | Balanced accuracy 0.979, p < 0.002 (permutation test, 500 permutations) | The 128-d embedding space strongly encodes recording-domain differences. A linear classifier separates TONAS from cante100 recordings near-perfectly, far above the null distribution maximum of 0.677. |
| **Palo probe** (logistic regression, 10 palos, cante100 only, LOO) | Balanced accuracy 0.243 (2.4× chance), p < 0.002 | Palo information is linearly decodable above chance, but weakly. The embedding is not primarily organised by palo (76% error rate on a balanced 10-class problem), but palo is not fully absent either. |
| **Silhouette score** (cosine, cantaor labels, pretrained vs. fine-tuned) | Pretrained: −0.2141 (full) / −0.1084 (cante100-only); Fine-tuned: −0.1220 (full) / **+0.0259** (cante100-only). All p < 0.001 (999-permutation test). | Fine-tuning increases cantaor cluster cohesion in both subsets. The only positive score (+0.0259, cante100-only) is the first metric-space confirmation of cantaor grouping independent of UMAP. The full-set gap (−0.1220 vs. +0.0259) reflects the domain confound depressing within-cantaor cohesion. |
| **Metric comparison: original vs. cante100-only pairs** | AUC 0.801 vs. 0.813 (Δ = −0.012); EER 32.1% vs. 25.4% (Δ = +6.7 pp); bootstrap 95% CIs fully overlapping | No statistically detectable difference given the available statistical power (see below). |

### The source confound: real at the representation level, undetectable at the metric level

The source probe result is genuine and consequential: the embedding encodes acoustic-domain
information strongly enough that TONAS recordings (a cappella, 8–33 s, mono WAV archival)
and cante100 recordings (guitar-accompanied, 75–420 s, MP3) are nearly perfectly
separable in the 128-d space. For the 10 cantaores present in both datasets, this means
the model's verification signal cannot be attributed purely to vocal style — acoustic-domain
recognition is a plausible contributor.

However, restricting the evaluation to cante100-only pairs (n = 44 positives, 1,069
negatives) yields AUC 0.813 and EER 25.4% — neither detectably different from the
original figures under bootstrap resampling (AUC Δ = −0.012, CIs [0.751, 0.848] vs.
[0.739, 0.881]; EER CIs [24.0%, 34.9%] vs. [17.5%, 35.6%]). The EER confidence interval
on the clean subset spans 18 percentage points ([17.5%, 35.6%]), reflecting the uncertainty
inherent in having only 44 positive pairs — that width is a direct statement about
statistical power, not evidence of equivalence.

The absence of a detectable metric shift is consistent with a cancellation mechanism:
cross-dataset pairs inflate the negative pool with easy-to-separate pairs (1,034 of 2,340
negatives cross the domain boundary) while simultaneously populating the positive pool with
hard-to-match pairs (21 of 78 positives span the TONAS/cante100 domain gap, which the
model tends to penalise). These opposing pressures are consistent with the near-zero net
shift in AUC, but this account should not be read as a demonstration that the confound is
harmless — it is a description of how two biases interact in the specific pair structure
of this evaluation, not a proof of their cancellation in general.

### Palo confound: present but limited

Within cante100-only same-cantaor pairs, same-palo pairs (n = 6) have a lower mean
cosine distance than cross-palo pairs (n = 38; 0.167 vs. 0.259), but the medians are
nearly identical (0.170 vs. 0.181) and the Mann-Whitney test is non-significant
(U = 92.0, p = 0.47). Three of the six same-palo pairs come from single-palo cantaores
for whom the comparison is trivial by construction; the genuinely multi-palo subset
reduces to n = 3. No conclusion about palo-invariance can be drawn from this analysis —
the sample is too small to distinguish a real effect from noise. The palo probe result
(balanced accuracy 0.243) remains the more informative signal: palo leaks into the
embedding space, but weakly.

### Summary for interpretation

The central result of this project — that a triplet-trained projection head over frozen
OpenL3 embeddings achieves AUC ≈ 0.80 on closed-set cantaor verification, a reproducible
improvement over the pretrained baseline — stands. The post-hoc diagnostics add two
qualifications that should accompany any interpretation:

1. **For the 10 cantaores present in both TONAS and cante100**, verification performance
   cannot be attributed solely to vocal style recognition. The embedding encodes
   recording-domain differences (source probe: 0.979 balanced accuracy), and cross-dataset
   positive pairs are a non-trivial fraction of the evaluation (21 of 78, 26.9%). The
   claim "the embedding captures cantaor interpretive identity" is better scoped to
   within-domain comparisons.

2. **Palo is a weak confound across the full dataset.** The embedding is not organised
   primarily by palo, but palo information is present above chance. For cantaores whose
   recordings in this dataset concentrate in a single palo (e.g. Antonio Piñana, Paco
   Toronjo), verification performance may reflect palo-level similarity alongside
   individual vocal style.

These limitations are inherent to the dataset's size and structure rather than to the
modelling approach, and are documented here as the basis for interpreting the results
honestly rather than as arguments against their validity.